# Hybrid GPR

This notebook trains a Gaussian process regressor to predict $-\Delta G_{abs}$. The prior is taken to be a linear function of total number of each bead type. Results are saved in the `hybridmodels` folder. SHAP values are also computed. There is an option to use full feature set or a reduced feature set. The reduced feature set is the max_[W] along with totals of each type of bead. The model in the publicaiton has the full dataset with ARD (a length scale for every dimension). The prior is also saved and treated as the "learned equation" in the paper.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error as mse
import GPy
import shap
import pickle
import functools
from IPython.display import display
from IPython.display import Image
from sklearn.linear_model import LinearRegression
import json
from matplotlib.pyplot import figure

 /Users/dja3/envir/ii/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning:IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


## Choose model type

In [3]:
ard = True  # True means use a length scale for every dimension
reduced = False  # True means means use only important ones

prefix = ""
if ard:
    prefix += "ard_"
if reduced:
    prefix += "red_"

## Get Data

In [4]:
features = [
    "num_[W]",
    "max_[W]",
    "num_[Tr]",
    "max_[Tr]",
    "num_[Ta]",
    "max_[Ta]",
    "num_[R]",
    "max_[R]",
    "[W]",
    "[Tr]",
    "[Ta]",
    "[R]",
    "rel_shannon",
    "length",
]

In [5]:
df = pd.read_csv("data/disp_dataset.csv")
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values * (-1)

In [6]:
if reduced:

    mask = np.zeros(len(df.columns) - 1).astype(bool)
    mask[8:12] = True

    X_adjust = np.multiply(X[:, mask], X[:, 13][:, np.newaxis])
    X_adjust = np.concatenate([X[:, 1].reshape(-1, 1), X_adjust], axis=1)

else:

    X_adjust = X

print(X_adjust)

[[ 0.          0.          0.4        ...  0.3         0.35616074
  40.        ]
 [ 0.          0.          0.375      ...  0.31578947  0.36186147
  38.        ]
 [ 0.2         2.          0.2        ...  0.35294118  0.36667336
  34.        ]
 ...
 [ 0.75        3.          0.         ...  0.16666667  0.39094785
  24.        ]
 [ 0.8         3.          0.2        ...  0.14285714  0.38324006
  28.        ]
 [ 1.          2.          0.         ...  0.16666667  0.39094785
  24.        ]]


## Train the models

In [7]:
for f in range(5):

    # Get training data and scale
    train_mask = np.genfromtxt(
        "data/data_split.csv", delimiter=",", skip_header=1, dtype=bool, usecols=f
    )
    scaler = StandardScaler()
    X_train = X_adjust[train_mask, :]
    X_test = X_adjust[np.logical_not(train_mask), :]
    y_train = y[train_mask]
    y_test = y[np.logical_not(train_mask)]

    X_train_scal = scaler.fit_transform(X_train)
    X_test_scal = scaler.transform(X_test)

    # Get the input for the prior
    mask = np.zeros(len(df.columns) - 1).astype(bool)
    mask[8:12] = True
    mask[9] = False

    X_train_prior = np.multiply(
        X[train_mask, :][:, mask], X[train_mask, 13][:, np.newaxis]
    )
    X_test_prior = np.multiply(
        X[np.logical_not(train_mask), :][:, mask],
        X[np.logical_not(train_mask), 13][:, np.newaxis],
    )

    # Learn the prior
    reg = LinearRegression().fit(X_train_prior, y_train)
    yprior_train = reg.predict(X_train_prior)
    yprior_test = reg.predict(X_test_prior)

    # Save the prior
    results = {"coef": list(reg.coef_), "intercept": reg.intercept_}
    with open(
        "hybridmodels/" + prefix + "linregprior" + "_fold" + str(f) + ".json", "w"
    ) as json_file:
        json.dump(results, json_file, indent=4)

    # Update training data
    ysubtract_train = y_train - yprior_train
    ysubtract_test = y_test - yprior_test

    # Build the GP model
    if reduced:
        input_dim = 5
    else:
        input_dim = 14

    kernel = GPy.kern.Matern52(input_dim=input_dim, ARD=ard)
    m = GPy.models.GPRegression(
        X_train_scal, ysubtract_train.reshape(-1, 1), kernel, normalizer=True
    )
    m[".*Gaussian_noise_*"] = 0.1

    # Train the model
    m.optimize_restarts(num_restarts=10)
    display(m)

    # Save the model
    with open("hybridmodels/" + prefix + "prior_fold" + str(f) + ".pkl", "wb") as file:
        pickle.dump(m, file)

Optimization restart 1/10, f = 3404.840331345386


 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:166: RuntimeWarning:overflow encountered in divide
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:137: RuntimeWarning:overflow encountered in square
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:138: RuntimeWarning:invalid value encountered in add
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:586: RuntimeWarning:invalid value encountered in multiply
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:589: RuntimeWarning:invalid value encountered in subtract


Optimization restart 2/10, f = 3410.072613277775
Optimization restart 3/10, f = 3404.8355883763597
Optimization restart 4/10, f = 3404.8393269708167
Optimization restart 5/10, f = 3490.057311320389
Optimization restart 6/10, f = 3404.8411110476272
Optimization restart 7/10, f = 3407.344330149238
Optimization restart 8/10, f = 3404.897528856927
Optimization restart 9/10, f = 3423.9674913867475
Optimization restart 10/10, f = 3404.830211264571


GP_regression.,value,constraints,priors
Mat52.variance,1.2805640399083045,+ve,
Mat52.lengthscale,"(14,)",+ve,
Gaussian_noise.variance,0.8609329908998499,+ve,


Optimization restart 1/10, f = 3400.950561666933
Optimization restart 2/10, f = 3400.9262341998206
Optimization restart 3/10, f = 3429.8463309677013
Optimization restart 4/10, f = 3400.395536605386
Optimization restart 5/10, f = 3535.240628287983


...

GP_regression.,value,constraints,priors
Mat52.variance,1.2072275011341607,+ve,
Mat52.lengthscale,"(14,)",+ve,
Gaussian_noise.variance,0.8599322977038953,+ve,


Optimization restart 1/10, f = 3390.420942841224
Optimization restart 2/10, f = 3410.736793333119
Optimization restart 3/10, f = 3390.461606437567
Optimization restart 4/10, f = 3390.7945405605283
Optimization restart 5/10, f = 3395.623664730813


 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:166: RuntimeWarning:overflow encountered in divide
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:137: RuntimeWarning:overflow encountered in square
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:138: RuntimeWarning:invalid value encountered in add
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:586: RuntimeWarning:invalid value encountered in multiply
 /Users/dja3/envir/ii/lib/python3.12/site-packages/GPy/kern/src/stationary.py:589: RuntimeWarning:invalid value encountered in subtract


Optimization restart 6/10, f = 3391.6320333293816
Optimization restart 7/10, f = 3390.9021234011907
Optimization restart 8/10, f = 3390.4311433503617
Optimization restart 9/10, f = 3390.7176701223243
Optimization restart 10/10, f = 3390.4428697578114


GP_regression.,value,constraints,priors
Mat52.variance,0.9775017330000024,+ve,
Mat52.lengthscale,"(14,)",+ve,
Gaussian_noise.variance,0.849438649880811,+ve,


## Look at the results

In [8]:
mse_df = pd.DataFrame(
    index=["train", "test", "train_prior", "test_prior"], columns=range(5)
)

for i in range(5):

    # Load model and test/train split
    with open("hybridmodels/" + prefix + "prior_fold" + str(i) + ".pkl", "rb") as file:
        m = pickle.load(file)
    train_mask = np.genfromtxt(
        "data/data_split.csv", delimiter=",", skip_header=1, dtype=bool, usecols=i
    )

    print(m.kern.lengthscale)

    # Get data
    scaler = StandardScaler()
    X_train = X_adjust[train_mask, :]
    X_test = X_adjust[np.logical_not(train_mask), :]
    y_train = y[train_mask]
    y_test = y[np.logical_not(train_mask)]

    X_train_scal = scaler.fit_transform(X_train)
    X_test_scal = scaler.transform(X_test)

    # Get data for input to prior
    mask = np.zeros(len(df.columns) - 1).astype(bool)
    mask[8:12] = True
    mask[9] = False
    X_train_prior = np.multiply(
        X[train_mask, :][:, mask], X[train_mask, 13][:, np.newaxis]
    )
    X_test_prior = np.multiply(
        X[np.logical_not(train_mask), :][:, mask],
        X[np.logical_not(train_mask), 13][:, np.newaxis],
    )

    # Get prior
    with open(
        "hybridmodels/" + prefix + "linregprior" + "_fold" + str(0) + ".json", "r"
    ) as file:
        linreg = json.load(file)
    coef = np.array(linreg["coef"])
    intercept = linreg["intercept"]

    yprior_train = np.sum(X_train_prior * coef, axis=1) + intercept
    yprior_test = np.sum(X_test_prior * coef, axis=1) + intercept

    # Get predictions
    yp_train, _ = m.predict(X_train_scal)
    yp_test, _ = m.predict(X_test_scal)

    yp_train += yprior_train.reshape(-1, 1)
    yp_test += yprior_test.reshape(-1, 1)

    mse_df.iloc[0, i] = mse(y_train, yp_train)
    mse_df.iloc[1, i] = mse(y_test, yp_test)
    mse_df.iloc[2, i] = mse(y_train, yprior_train)
    mse_df.iloc[3, i] = mse(y_test, yprior_test)

  index  |  GP_regression.Mat52.lengthscale  |  constraints  |  priors
  [0]    |                       3.95769082  |      +ve      |        
  [1]    |                       4.04941463  |      +ve      |        
  [2]    |                      15.15418957  |      +ve      |        
  [3]    |                    4616.26202018  |      +ve      |        
  [4]    |                    4420.61782463  |      +ve      |        
  [5]    |                      16.51560031  |      +ve      |        
  [6]    |                    4154.92799527  |      +ve      |        
  [7]    |                      13.74105931  |      +ve      |        
  [8]    |                       5.60073118  |      +ve      |        
  [9]    |                       5.18168557  |      +ve      |        
  [10]   |                    3299.01379094  |      +ve      |        
  [11]   |                    4266.58235375  |      +ve      |        
  [12]   |                    4061.86776760  |      +ve      |        
  [13]

In [9]:
mse_df.to_csv("hybridmodels/" + prefix + "prior_mse.csv")
mse_df.head()

,0,1,2,3,4
train,0.300752,0.308423,0.309822,0.313893,0.31394
test,0.360524,0.315334,0.322705,0.304848,0.303289
train_prior,0.355204,0.369688,0.36492,0.371418,0.376504
test_prior,0.416919,0.358982,0.378055,0.35206,0.331717


## Compute SHAP

### Non-linear SHAP

In [10]:
def m_mu(X, model):
    mu, var = model.predict(X)
    return mu.flatten()

In [11]:
for i in range(1):

    # Load model and test/train split
    with open("hybridmodels/" + prefix + "prior_fold" + str(i) + ".pkl", "rb") as file:
        m = pickle.load(file)
    train_mask = np.genfromtxt(
        "data/data_split.csv", delimiter=",", skip_header=1, dtype=bool, usecols=i
    )

    # Get data
    scaler = StandardScaler()
    X_train = X_adjust[train_mask, :]
    X_test = X_adjust[np.logical_not(train_mask), :]

    X_train_scal = scaler.fit_transform(X_train)
    X_test_scal = scaler.transform(X_test)

    # Compute SHAP values
    explainer = shap.KernelExplainer(
        functools.partial(m_mu, model=m),
        shap.kmeans(X_train_scal, 40),
        feature_names=features,
    )
    shap_values = explainer(X_test_scal)

    # Save SHAP results
    with open("hybridmodels/shap_values_" + prefix + str(i) + ".pkl", "wb") as f:
        pickle.dump(shap_values, f)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 625/625 [54:59<00:00,  5.28s/it]


In [12]:
# print(shap_values)

### Linear SHAP

In [13]:
def linearshap(coeffs, intercept, x):
    """Compute SHAP values analytically for a linear equation."""

    # determine predictions
    y = np.matmul(x, coeffs) + intercept

    # determine mean
    ymean = y.mean()

    # compute shap
    xx = np.transpose(np.transpose(x) - x.mean(axis=0).reshape(-1, 1))
    shapvalues = np.multiply(coeffs, xx)

    # base values
    shapbase = y * 0 + ymean

    shapex = shap.Explanation(values=shapvalues, base_values=shapbase, data=x)

    return shapex

In [14]:
for i in range(1):

    train_mask = np.genfromtxt(
        "data/data_split.csv", delimiter=",", skip_header=1, dtype=bool, usecols=i
    )

    # Get data for input to prior
    mask = np.zeros(len(df.columns) - 1).astype(bool)
    mask[8:12] = True
    mask[9] = False
    X_train_prior = np.multiply(
        X[train_mask, :][:, mask], X[train_mask, 13][:, np.newaxis]
    )
    X_test_prior = np.multiply(
        X[np.logical_not(train_mask), :][:, mask],
        X[np.logical_not(train_mask), 13][:, np.newaxis],
    )

    # Get prior
    with open(
        "hybridmodels/" + prefix + "linregprior" + "_fold" + str(0) + ".json", "r"
    ) as file:
        linreg = json.load(file)
    coef = np.array(linreg["coef"])
    intercept = linreg["intercept"]

    # Compute linear SHAP values
    shap_lin_values = linearshap(coef, intercept, X_test_prior)

    # Save SHAP results
    with open("hybridmodels/shap_lin_values_" + prefix + str(i) + ".pkl", "wb") as f:
        pickle.dump(shap_lin_values, f)

In [16]:
# print(shap_lin_values)